# Attacks on quantum backends through qBraid

*Builds on `3a_bb84_grid_link.ipynb`. Each attack is one BB84 circuit: 12 rounds, plus 12 "Eve" qubits when she's present. That makes one job per attack per backend.*

| Attack | What Eve does | What the link should see |
|---|---|---|
| quiet | nothing | QBER = line noise |
| tap | intercept-resend on every round | QBER ≈ 25% |
| partial | intercept-resend on 3 of 12 rounds | QBER between quiet and tap; may stay under 11% |
| blinding | taps every round and controls Bob's detectors, so he only gets a result when his basis matches hers | QBER stays low (**the error check is fooled**), but half the rounds don't click: a **click-rate monitor** catches it |
| dos | scrambles every qubit (noise injection) | QBER ≈ 50%: the link aborts and falls back to PQC keys, with no outage |

The random bits and bases come from a **quantum random number generator (QRNG) run on qBraid**, so the key's randomness comes from physics too. This is the README's warm-up stretch goal.

> **Kernel:** qBraid-SDK (v0.12.2). For the Quantinuum emulator, accept Quantinuum's terms once under Devices.

In [ ]:
import importlib.util
if importlib.util.find_spec("qbraid") is None:
    raise ImportError("Wrong kernel: pick qBraid-SDK (v0.12.2) at the top right, then run this cell again.")
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2
import sys, json, numpy as np
sys.path.insert(0, "network_design")
from qiskit import QuantumCircuit
from hw_attacks import make_round_choices, circuit, score, qrng_bits
from qbraid.runtime import QbraidProvider
import warnings; warnings.filterwarnings("ignore", message="Circuit contains measurement gates")
provider = QbraidProvider(); R = 12; RUNS = []          # run record: (what, device, job id, shots, status, result)

## 1. Quantum random bits from qBraid

A 12-qubit circuit with H then measure, run for 3 shots, gives 36 random bits: Alice's bits, Alice's bases and Bob's bases.

In [ ]:
rng_qc = QuantumCircuit(R, R); rng_qc.h(range(R)); rng_qc.measure(range(R), range(R))
QRNG_DEVICE = "qbraid:qbraid:sim:qir-sv"      # free; to use a real QPU's randomness, swap in a hardware device (credits)
job = provider.get_device(QRNG_DEVICE).run(rng_qc, shots=3)
bits = qrng_bits(job.result().data.get_counts(), R); RUNS.append(("QRNG bits", QRNG_DEVICE, job.id, 3, str(job.status()), "36 random bits"))
choices = make_round_choices(R, seed=0, bits=bits)
print("Alice bits :", bits[0]); print("Alice bases:", bits[1]); print("Bob bases  :", bits[2])
print("rounds where bases match (sifted):", int((bits[1] == bits[2]).sum()), "of", R)

## 2. Check every attack locally first (Aer, 2.5% line noise, free and instant)

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError
nm = NoiseModel(); nm.add_all_qubit_readout_error(ReadoutError([[0.975, 0.025], [0.025, 0.975]]))
aer = AerSimulator(method="stabilizer", noise_model=nm)
ATTACKS = ["quiet", "tap", "partial", "blinding", "dos"]
circuits = {a: circuit(choices, a, frac=0.25, seed=1) for a in ATTACKS}
for a, (qc, tapped) in circuits.items():
    r = score(aer.run(qc, shots=200).result().get_counts(), choices, a, tapped)
    print(f"{a:9s} QBER {r['qber']:.3f}  click yield {r['click_yield']:.2f}  ({r['sifted_rounds']} sifted rounds)")

## 3. Run every attack on qBraid backends

Free simulators run by default. **Real hardware:** uncomment one QPU line. The Open Quantum (`openquantum:`) devices are the same route Step 1 used for the Forte run (accept Open Quantum's terms on qBraid first). Circuits too big for a device are skipped automatically:

| Circuit | Qubits |
|---|---|
| quiet, dos | 12 |
| partial | 15 |
| tap, blinding (same circuit) | 24 |

In [ ]:
BACKENDS = [
    ("qbraid:qbraid:sim:qir-sv", 30, {}),                                           # free, noise-free
    ("ionq:ionq:sim:simulator", 29, {"runtime_options": {"noise_model": "aria-1"}}), # free, IonQ gate noise
    ("quantinuum:quantinuum:sim:h2-emulator", 56, {}),                              # free emulator (accept Quantinuum terms first)
    # --- REAL HARDWARE: uncomment ONE, after checking with your team / a mentor ---
    # ("openquantum:ionq:qpu:forte-1", 36, {}),            # all 5 attacks fit; Step 1 already ran here
    # ("openquantum:iqm:qpu:emerald", 54, {}),             # all 5 fit (superconducting: compare noise with Forte)
    # ("openquantum:iqm:qpu:garnet", 20, {}),              # quiet, partial, dos only
    # ("openquantum:aqt:qpu:ibex-q1", 12, {}),             # quiet and dos only
]
results = {}
for dev_id, max_q, opts in BACKENDS:
    dev = provider.get_device(dev_id)
    for a, (qc, tapped) in circuits.items():
        if a == "blinding" and (dev_id, "tap") in results:
            # blinding uses the same circuit as tap; only the scoring differs, so reuse the tap job (saves a run)
            r = score(results[(dev_id, "tap")]["counts"], choices, "blinding", tapped)
            results[(dev_id, a)] = {**r, "counts": results[(dev_id, "tap")]["counts"]}
            print(f"{dev_id:38s} {a:9s} QBER {r['qber']:.3f}  click yield {r['click_yield']:.2f}  (scored from the tap job)"); continue
        if qc.num_qubits > max_q:
            print(f"{dev_id:38s} {a:9s} skipped: needs {qc.num_qubits} qubits"); continue
        try:
            job = dev.run(qc, shots=200, **opts); counts = job.result().data.get_counts()
            r = score(counts, choices, a, tapped); results[(dev_id, a)] = {**r, "counts": counts}
            RUNS.append((f"3f {a}", dev_id, job.id, 200, str(job.status()), f"QBER {r['qber']:.3f}, yield {r['click_yield']:.2f}"))
            print(f"{dev_id:38s} {a:9s} QBER {r['qber']:.3f}  click yield {r['click_yield']:.2f}  job {job.id}")
        except Exception as e:
            RUNS.append((f"3f {a}", dev_id, "-", 200, f"failed: {type(e).__name__}: {str(e)[:80]}", "")); print(dev_id, a, "failed:", e)

## 4. Alarm decisions

- **QBER alarm:** QBER above the quiet run's QBER + 3σ (or above 11%, where BB84 aborts).
- **Click alarm:** fewer than 80% of matched-basis rounds produced a result (the click-rate monitor).

A blinding attack should pass the QBER check and trip only the click alarm. That's why real systems need detector-side countermeasures.

In [ ]:
for dev_id, _, _ in BACKENDS:
    if (dev_id, "quiet") not in results: continue
    q0 = results[(dev_id, "quiet")]; n = max(1, q0["sifted_rounds"] * 200)
    thr = max(q0["qber"], 0.0) + 3 * np.sqrt(max(q0["qber"], 0.01) * (1 - max(q0["qber"], 0.01)) / n)
    print(dev_id, f"(quiet QBER {q0['qber']:.3f}, alarm above {thr:.3f})")
    for a in ATTACKS:
        r = results.get((dev_id, a))
        if r: print(f"   {a:9s} QBER alarm: {str(r['qber'] > thr or r['qber'] > 0.11):5s}  click alarm: {str(r['click_yield'] < 0.8):5s}  abort (QBER > 11%): {r['qber'] > 0.11}")
print("\nRun record (paste into WRITEUP.md):")
for row in RUNS: print(" | ".join(map(str, row)))